In [ ]:
# Source - https://stackoverflow.com/a
# Posted by G M, modified by community. See post 'Timeline' for change history
# Retrieved 2026-01-12, License - CC BY-SA 4.0
if 'google.colab' in str(get_ipython()):
  !git clone https://github.com/Vladislavicious/jenga_ml.git
  %cd jenga_ml
  !git switch dev
  !pip install -r requirements.txt
else:
  print('Not running on CoLab')

Not running on CoLab


In [ ]:
import random
from environment import make_jenga_env
from model_trainer import *
import numpy as np
import time


In [ ]:
!uv pip install "git+https://github.com/google-deepmind/mujoco_warp.git"

In [ ]:
import os
import subprocess
import numpy as np
import warp as wp
import mujoco_warp as mjw

# Set up GPU rendering.
if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.')

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Configuring MuJoCo for GPU rendering, setting ', end="")
%env MUJOCO_GL=egl

# avoid warp output in cells
wp.config.quiet = True

try:
  print('Validating MuJoCo Warp installation: ', end="")
  import mujoco
  import mujoco_warp as mjw

  _MJCF=r"""
  <mujoco>
    <worldbody>
      <body>
        <freejoint/>
        <geom size=".15" mass="1" type="sphere"/>
      </body>
    </worldbody>
  </mujoco>
  """

  mjm = mujoco.MjModel.from_xml_string(_MJCF)
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".')

print('success.')

In [ ]:
n_blocks = 6
random.seed(123)
np.random.seed(123)

env = make_jenga_env(n_blocks=n_blocks, render=True)

In [ ]:
step_count = 250
iterations = 20
total_steps = step_count * iterations

trainer = JengaML_Trainer(env, blocks_count=n_blocks, total_timesteps=total_steps, n_steps=step_count)

In [ ]:
env.reset()
trainer.train()

In [ ]:
trainer.evaluate(max_steps=step_count, visualize=True)

In [ ]:
extended_trainer = JengaML_Trainer(env, blocks_count=n_blocks, total_timesteps=total_steps, n_steps=step_count)
extended_trainer.load_trained_model(FINAL_MODEL_PATH)
extended_trainer.train()

In [ ]:
env.render()

In [ ]:
trainer.load_trained_model(FINAL_MODEL_PATH)
trainer.evaluate(2500, True)